### Retrieval-Augmented Generation (RAG) - Practice & Exercises
In this notebook, you will practice building a RAG pipeline by chunking text, embedding pages, and retrieving context.

#### Part 1: Follow-Along Practice
##### GoalLoad a text sample, split it, index it into a Chroma database, and construct a RAG chain to answer queries.

##### Steps:1. Define split text and index in a local Chroma store.
2. Embed split document using `OpenAIEmbeddings`.
3. Setup retriever and chain using `create_stuff_documents_chain` or custom formatting.
4. Invoke and answer context questions.

In [ ]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

docs = [
    Document(page_content="LangChain was launched in October 2022 as an open-source project by Harrison Chase."),
    Document(page_content="LangGraph was introduced to support cyclic graph architectures.")
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma.from_documents(docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 1})

model = ChatOpenAI(model="gpt-4o")
prompt = ChatPromptTemplate.from_template("Context:\n{context}\n\nQuestion: {question}\n\nAnswer:")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

print(rag_chain.invoke("When was LangChain launched?"))

#### Part 2: Practice Exercises
##### Exercise 1: Conversational RAG with Memory
##### GoalBuild a conversational RAG chain that answers user queries based on document context, remembering previous questions and answers in a history list.

##### Sample InputSequential prompts:
1. `"Who launched LangChain?"`
2. `"When did he launch it?"`

##### Sample Output1. `"LangChain was launched by Harrison Chase."`
2. `"He launched it in October 2022."`

##### Plan1. **Define Conversation Prompt:** Create a prompt that accepts both `context` and `chat_history` variables.
2. **Format Chain:** Format a custom helper function that joins previous conversation strings, retrieves database doc content, and invokes the model.
3. **Run sequentially:** Verify context retrieval and coreferencing memory.

In [ ]:
# TODO: Implement conversational RAG memory helper here


<details>
<summary>💡 Click to see solution</summary>

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o")
chat_history = []

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's question using the context. Context: {context}"),
    ("placeholder", "{chat_history}"),
    ("human", "{question}")
])

def conversational_rag(question: str):
    ## Retrieve matching documents    docs = retriever.invoke(question)
    context_text = "\n".join(d.page_content for d in docs)
    
    ## Prepare messages    formatted = rag_prompt.format_messages(
        context=context_text,
        chat_history=chat_history,
        question=question
)
    res = model.invoke(formatted)
    
    ## Update history    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=res.content))
    
    print(f"\nAI: {res.content}")

conversational_rag("Who launched LangChain?")
conversational_rag("When did he launch it?")
```
</details>